**Parte 0**: Connessione Google Drive a Google Colab, al fine di poter utilizzare il dataset, e import delle librerie necessarie

In [10]:
!pip install pgmpy
!apt-get install -y swi-prolog
!pip install pyswip
import os
import zipfile
import urllib.request
import pandas as pd
import datetime
import numpy as np
import warnings
from os import system

# Librerie per l'albero decisionale
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Librerie per la rete neurale
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# Librerie per la rete bayesiana
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
from sklearn.metrics import accuracy_score

# Librerie per prolog
from pyswip import Prolog

# Librerie per le ottimizzazioni dei parametri
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swi-prolog is already the newest version (8.4.2+dfsg-2ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


**Parte 1**: Caricamento del dataset HomeC.cvs, pulizia di questo sistemando errori e feature engineering (ovvero creazione caratteristiche)

In [6]:
url = "https://github.com/MeliotaLucia/EnergySafe/raw/main/HomeC.zip"
zip_path = "HomeC.zip"
if not os.path.exists(zip_path):
    print("Scaricamento del dataset in corso...")
    urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(".")
    print("Dataset estratto correttamente!")
df = pd.read_csv('HomeC.csv')

warnings.filterwarnings("ignore", category=FutureWarning)
# Controlli generali del dataset: numero righe e colonne, visualizzazione delle righe e se ci sono valori mancanti
print(f"Il dataset ha {df.shape[0]} righe e {df.shape[1]} colonne.")
print(df.columns)
print(df.isnull().sum())

# Risoluzione di problemi trovati nel dataset:
# - l'ultima riga è vuota
df = df.dropna()

# - alcuni input in time sono non interi, dunque conversione di qualsiasi stringa in numero.
df['time'] = pd.to_numeric(df['time'], errors='coerce') #se non è una stringa che in realtà è un numero, diventa NaN
df = df.dropna(subset=['time']) #eliminazione il NaN
df['time'] = df['time'].astype(int)
df['time'] = pd.to_numeric(df['time'])

# - trasformazione del tempo nel formato corretto
df['dt_time'] = pd.to_datetime(df['time'], unit='s')
df['hour'] = df['dt_time'].dt.hour
df['month'] = df['dt_time'].dt.month
df['is_weekend'] = df['dt_time'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)

# Crezione di una colonna "fascia_prezzo" che riguarda il risaprmio energetico, che sono tre fasce: costa meno, costa medio, costa troppo - queste fasce sono quelle reali
def assegna_fascia(row):
    ora = row['dt_time'].hour
    giorno = row['dt_time'].dayofweek # 0=Lunedì, 6=Domenica
    # Fascia F3: Domenica tutta la giornata o ogni giorno dalle 23 alle 7
    if giorno == 6 or ora >= 23 or ora < 7:
        return 'F3'
    # Fascia F2: Sabato (7-23) o feriali (7-8 e 19-23)
    if giorno == 5:
        if 7 <= ora < 23:
            return 'F2'
        else:
            return 'F3' # Sabato prima delle 7 o dopo le 23 è F3
    else: # Giorni feriali (Lunedì-Venerdì)
        if (7 <= ora < 8) or (19 <= ora < 23):
            return 'F2'
        elif 8 <= ora < 19:
            return 'F1'
        else:
            return 'F3'
# Applicazione della funzione al dataset
df['fascia_oraria'] = df.apply(assegna_fascia, axis=1)
print(df['fascia_oraria'].value_counts()) # per vedere se ho abbastanza diversità nel dataset per le varie fasce e si: 42% dei dati in F3, il 32% in F1 e il 26% in F2
# Trasformazione la fascia_oraria in numeri
df['fascia_num'] = df['fascia_oraria'].map({'F1': 1, 'F2': 2, 'F3': 3})

# Creazione delle 3 categorie di consumo: 0 (carico basso), 1 (carico medio), 2 (carico alto)
# Divisione in base ai kW, le tre categorie di consumo
df['target_carico'] = pd.qcut(df['use [kW]'], q=3, labels=[0, 1, 2])

# Visualizzazione dei limiti di consumo per ogni categoria
tabella_intervalli = df.groupby('target_carico', observed=True)['use [kW]'].agg(['min', 'max']).reset_index()
tabella_intervalli.columns = ['Classe di Carico', 'Consumo Min (kW)', 'Consumo Max (kW)']
tabella_intervalli['Classe di Carico'] = ['0 (Basso)', '1 (Medio)', '2 (Alto)']
print("RIASSUNTO SOGLIE DI CONSUMO:")
display(tabella_intervalli)

# Media mobile del consumo, prende anche i due valori precedenti a questo per avere appunto la memoria, senza picchi a caso
df['use_MA_3'] = df['use [kW]'].rolling(window=3).mean().fillna(0) # questo prende la media delle scorse 3 volte

Dataset estratto correttamente!


/tmp/ipykernel_933/3303246122.py:9: DtypeWarning: Columns (0,27) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('HomeC.csv')


Il dataset ha 503911 righe e 32 colonne.
Index(['time', 'use [kW]', 'gen [kW]', 'House overall [kW]', 'Dishwasher [kW]',
       'Furnace 1 [kW]', 'Furnace 2 [kW]', 'Home office [kW]', 'Fridge [kW]',
       'Wine cellar [kW]', 'Garage door [kW]', 'Kitchen 12 [kW]',
       'Kitchen 14 [kW]', 'Kitchen 38 [kW]', 'Barn [kW]', 'Well [kW]',
       'Microwave [kW]', 'Living room [kW]', 'Solar [kW]', 'temperature',
       'icon', 'humidity', 'visibility', 'summary', 'apparentTemperature',
       'pressure', 'windSpeed', 'cloudCover', 'windBearing', 'precipIntensity',
       'dewPoint', 'precipProbability'],
      dtype='object')
time                   0
use [kW]               1
gen [kW]               1
House overall [kW]     1
Dishwasher [kW]        1
Furnace 1 [kW]         1
Furnace 2 [kW]         1
Home office [kW]       1
Fridge [kW]            1
Wine cellar [kW]       1
Garage door [kW]       1
Kitchen 12 [kW]        1
Kitchen 14 [kW]        1
Kitchen 38 [kW]        1
Barn [kW]             

,Classe di Carico,Consumo Min (kW),Consumo Max (kW)
0,0 (Basso),0.000000,0.433000
1,1 (Medio),0.433017,0.808633
2,2 (Alto),0.808650,14.714567


**Parte 2**: Machine Learning - Alberi Decisionali (classificare il "Livello di Carico" della casa (Basso, Medio, Alto))


In [7]:
# Selezione delle features e creazione di X e Y
features = ['temperature', 'humidity', 'hour', 'is_weekend', 'fascia_num', 'use_MA_3']
X = df[features]
y = df['target_carico']

# Divisione del training set e del test set in 80% e 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Definizione della griglia dei parametri da esplorare
param_grid_dt = {
    'criterion': ['gini', 'entropy'],      # Metodo per valutare la purezza del nodo
    'max_depth': [3, 5, 7, 10, 15],        # Testiamo diverse profondità per evitare overfitting
    'min_samples_split': [2, 10, 20]       # Minimo numero di campioni per dividere un nodo
}

# Inizializzazione del classificatore base
dt_base = DecisionTreeClassifier(random_state=42)

# Impostazione la GridSearchCV (con Cross-Validation a 5 fold)
print("Avvio GridSearchCV per l'Albero Decisionale (Ricerca parametri + CV)...")
grid_search_dt = GridSearchCV(estimator=dt_base,
                              param_grid=param_grid_dt,
                              cv=5,
                              scoring='accuracy',
                              n_jobs=-1, # n_jobs=-1 dice al computer di usare tutti i processori disponibili per velocizzare
                              verbose=1)

# Addestramento della griglia sul training set
grid_search_dt.fit(X_train, y_train)

# Estrazione del modello migliore e i suoi risultati statistici
best_dt = grid_search_dt.best_estimator_
best_idx = grid_search_dt.best_index_
cv_mean = grid_search_dt.cv_results_['mean_test_score'][best_idx]
cv_std = grid_search_dt.cv_results_['std_test_score'][best_idx]

print(f"\n{'-' * 60}")
print("RISULTATI OTTIMIZZAZIONE ALBERO DECISIONALE: ")
print(f"Migliori iperparametri trovati: {grid_search_dt.best_params_}")
print(f"Accuratezza Media in CV (K-Fold=5): {cv_mean:.2%}")
print(f"Deviazione Standard in CV: {cv_std:.2%}")
print(f"{'-' * 60}\n")

# Valutazione finale del miglior modello sul test set
y_pred_dt = best_dt.predict(X_test)
print("PRESTAZIONI DEL MODELLO OTTIMIZZATO SUL TEST SET FINALE: ")
report_dict = classification_report(y_test, y_pred_dt, target_names=['Basso', 'Medio', 'Alto'], output_dict=True)
report_df = pd.DataFrame(report_dict).transpose().head(3)
report_df.columns = ['Precisione', 'Recall', 'F1-Score', 'Numero di righe (Esempi)']

# Formattazione per una visualizzazione migliore
tabella_percentuale = report_df.style.format({
    'Precisione': '{:.2%}',
    'Recall': '{:.2%}',
    'F1-Score': '{:.2%}',
    'Numero di righe (Esempi)': '{:.0f}'
})
display(tabella_percentuale)
print(f"\nAccuratezza Totale sul Test Set: {accuracy_score(y_test, y_pred_dt):.2%}")

Avvio GridSearchCV per l'Albero Decisionale (Ricerca parametri + CV)...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

------------------------------------------------------------
RISULTATI OTTIMIZZAZIONE ALBERO DECISIONALE: 
Migliori iperparametri trovati: {'criterion': 'gini', 'max_depth': 3, 'min_samples_split': 2}
Accuratezza Media in CV (K-Fold=5): 91.75%
Deviazione Standard in CV: 0.05%
------------------------------------------------------------

PRESTAZIONI DEL MODELLO OTTIMIZZATO SUL TEST SET FINALE: 


,Precisione,Recall,F1-Score,Numero di righe (Esempi)
Basso,95.68%,92.69%,94.16%,33823
Medio,87.63%,88.62%,88.12%,33600
Alto,92.14%,94.01%,93.07%,33359



Accuratezza Totale sul Test Set: 91.77%


**Parte 3**: Rete Neurale

In [9]:
# Creazione di un campionamento casuale dal dataframe di 100.000 righe per poter abbassare i tempi di computazione
df_sample = df.sample(n=100000, random_state=42)
X_sample = df_sample[features]
y_sample = df_sample['target_carico']

# Dividiamo in Train e Test sul campione. L'utilizzo di stratify è dovuto per mantenere le proporzioni
X_train_mlp, X_test_mlp, y_train_mlp, y_test_mlp = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42, stratify=y_sample
)

# Standardizzazione dei dati
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_mlp)
X_test_scaled = scaler.transform(X_test_mlp)

# Definizione della griglia dei parametri
param_grid_mlp = {
    'hidden_layer_sizes': [(64, 32), (100, 50)], # due architetture da due hidden layer
    'activation': ['relu', 'tanh'],              # diverse funzioni di attivazione
    'early_stopping': [True]                     # l'early stopping per evitare overfitting
}

# Inizializziazione dell'MLP
mlp_base = MLPClassifier(max_iter=200, random_state=42)

# Impostazione della GridSearchCV (con Cross-Validation a 5 fold)
print("Avvio GridSearchCV per la Rete Neurale (MLP)...")
grid_search_mlp = GridSearchCV(estimator=mlp_base,
                               param_grid=param_grid_mlp,
                               cv=5,
                               scoring='accuracy',
                               n_jobs=-1,
                               verbose=2)

# Addestramento della griglia
grid_search_mlp.fit(X_train_scaled, y_train_mlp)

# Estrazione dei risultati statistici
best_mlp = grid_search_mlp.best_estimator_
best_idx_mlp = grid_search_mlp.best_index_
cv_mean_mlp = grid_search_mlp.cv_results_['mean_test_score'][best_idx_mlp]
cv_std_mlp = grid_search_mlp.cv_results_['std_test_score'][best_idx_mlp]

print(f"\n{'-' * 60}")
print("RISULTATI OTTIMIZZAZIONE RETE NEURALE: ")
print(f"Migliori iperparametri trovati: {grid_search_mlp.best_params_}")
print(f"Accuratezza Media in CV (K-Fold=5): {cv_mean_mlp:.2%}")
print(f"Deviazione Standard in CV: {cv_std_mlp:.2%}")
print(f"{'-' * 60}\n")

# alutazione finale del miglior modello sul Test Set
y_pred_mlp = best_mlp.predict(X_test_scaled)

print("PRESTAZIONI DELLA RETE NEURALE OTTIMIZZATA SUL TEST SET: ")
report_mlp = classification_report(y_test_mlp, y_pred_mlp, target_names=['Basso', 'Medio', 'Alto'], output_dict=True)
report_mlp_df = pd.DataFrame(report_mlp).transpose().head(3)
report_mlp_df.columns = ['Precisione', 'Recall', 'F1-Score', 'Esempi']
report_mlp_df['Esempi'] = report_mlp_df['Esempi'].astype(int)

display(report_mlp_df.style.format({'Precisione': '{:.2%}', 'Recall': '{:.2%}', 'F1-Score': '{:.2%}', 'Esempi': '{:d}'}))
print(f"\nAccuratezza Totale sul Test Set: {accuracy_score(y_test_mlp, y_pred_mlp):.2%}")

Avvio GridSearchCV per la Rete Neurale (MLP)...
Fitting 5 folds for each of 4 candidates, totalling 20 fits

------------------------------------------------------------
RISULTATI OTTIMIZZAZIONE RETE NEURALE: 
Migliori iperparametri trovati: {'activation': 'tanh', 'early_stopping': True, 'hidden_layer_sizes': (64, 32)}
Accuratezza Media in CV (K-Fold=3): 91.37%
Deviazione Standard in CV: 0.19%
------------------------------------------------------------

PRESTAZIONI DELLA RETE NEURALE OTTIMIZZATA SUL TEST SET: 


,Precisione,Recall,F1-Score,Esempi
Basso,94.88%,92.94%,93.90%,6715
Medio,87.74%,86.63%,87.18%,6666
Alto,91.08%,94.12%,92.58%,6619



Accuratezza Totale sul Test Set: 91.23%


**Parte 4** : Rete bayesiana

Si è osservato che le variabili ambientali (meteo e temperatura) da sole non sono sufficienti a determinare univocamente il carico energetico, portando a una distribuzione di probabilità quasi uniforme (circa 33% per ogni classe). Il modello acquisisce invece un'elevata confidenza predittiva solo quando viene integrato il contesto del trend di consumo recente, dimostrando che il comportamento dell'utente prevale sulle condizioni atmosferiche.<br> Purtroppo per quanto si sarebbe voluto aggiungere la presenza umana come cosa principale dei consumi, si è scelto di non includerla come variabile stimata per evitare di introdurre bias deterministici nel modello. Una stima della presenza basata su regole fisse renderebbe il sistema rigido; pertanto, si è preferito lasciare che la rete apprendesse le correlazioni dirette tra variabili ambientali e carico energetico, mantenendo l'integrità statistica dei dati osservati. Quindi si userà questa rete bayesiana per prevedere il carico e di conseguenza, in base alla KB, scegliere cosa spegnere, ritardare o meno.

In [13]:
def visualizza_risultati(q):
    mapping = {0: 'Basso', 1: 'Medio', 2: 'Alto'}
    valori = q.values

    print(f"{'Stato Carico':<15} | {'Probabilità':<10}")
    print("-" * 30)

    for i, prob in enumerate(valori):
        nome_classe = mapping.get(i, f"Classe {i}")
        print(f"{nome_classe:<15} | {prob:.2%}")

    print("\n")

In [15]:
# Creazione una copia del dataset per la rete bayesiana
df_bayes = df.copy()

# Inizio fase di Data Preprocessing: divisione in categorie piuttosto che numeri continui.
# Temperatura in 3 livelli, l'umidità in tre livelli, media degli ultimi consumi, l'orario e se sta piovendo o meno
df_bayes['temp_cat'] = pd.qcut(df['temperature'], q=3, labels=['Freddo', 'Mite', 'Caldo'])
df_bayes['umid_cat'] = pd.qcut(df['humidity'], q=3, labels=['Secco', 'Normale', 'Umido'])
df_bayes['media_mobile_cat'] = pd.qcut(df['use_MA_3'], q=3, labels=['Trend_Basso', 'Trend_Medio', 'Trend_Alto'])
df_bayes['hour_cat'] = pd.cut(df_bayes['hour'], bins=[0, 6, 12, 18, 24], labels=['Notte', 'Mattina', 'Pomeriggio', 'Sera'], include_lowest=True)
df_bayes['pioggia_cat'] = pd.cut(df['precipProbability'], bins=[-0.1, 0.1, 0.5, 1.1], labels=['No_Pioggia', 'Prob_Bassa', 'Prob_Alta'])
# Omissione delle colonne non interessanti
colonne_df_bayes = ['pioggia_cat', 'fascia_oraria', 'temp_cat', 'umid_cat', 'media_mobile_cat', 'hour_cat', 'target_carico']
df_bayes = df_bayes[colonne_df_bayes].copy()

# Creazione di un campione di 100.000 righe per coerenza con l'MLP
df_bayes_sample = df_bayes.sample(n=100000, random_state=42).reset_index(drop=True)

# Definizione la variabile target e i nodi in input
target = 'target_carico'
nodi_input = [nodo for nodo in model.nodes() if nodo != target]

# Impostazione del K-Fold a 5
kf = KFold(n_splits=5, shuffle=True, random_state=42)
accuratezze_bayes = []

print("Avvio Cross-Validation (5-Fold) per la Rete Bayesiana...")

# Cross validation manuale a 5 fold
for fold_idx, (train_index, test_index) in enumerate(kf.split(df_bayes_sample)):
    # Definizione del DAG
    model = DiscreteBayesianNetwork([
        ('hour_cat', 'target_carico'),
        ('pioggia_cat', 'target_carico'),
        ('umid_cat', 'target_carico'),
        ('fascia_oraria', 'target_carico'),
        ('temp_cat', 'target_carico'),
        ('media_mobile_cat', 'target_carico')
    ])

    # Divisione in Training e Test per il fild i-esiom
    train_data = df_bayes_sample.iloc[train_index]
    test_data = df_bayes_sample.iloc[test_index]

    # Addestramento del modello sul training set
    model.fit(train_data)

    # Predizione sul test_data (nascondendo la variabile target)
    X_test_fold = test_data[nodi_input]
    y_true_fold = test_data[target].values

    predictions = model.predict(X_test_fold)
    y_pred_fold = predictions[target].values

    # Calcolo dell'accuratezza del fold
    acc = accuracy_score(y_true_fold, y_pred_fold)
    accuratezze_bayes.append(acc)
    print(f"Completato Fold {fold_idx + 1}/5: Accuratezza {acc:.2%}")

# Riaddestramento della rete su tutti i dati
model = DiscreteBayesianNetwork([
        ('hour_cat', 'target_carico'),
        ('pioggia_cat', 'target_carico'),
        ('umid_cat', 'target_carico'),
        ('fascia_oraria', 'target_carico'),
        ('temp_cat', 'target_carico'),
        ('media_mobile_cat', 'target_carico')
    ])
model.fit(df_bayes_sample)

# Calcolo della media e della deviazione standard
media_bayes = np.mean(accuratezze_bayes)
std_bayes = np.std(accuratezze_bayes)

print(f"\n{'-' * 60}")
print("RISULTATI CROSS-VALIDATION RETE BAYESIANA: ")
print(f"Accuratezza Media in CV (K-Fold=5): {media_bayes:.2%}")
print(f"Deviazione Standard in CV: {std_bayes:.2%}")
print(f"{'-' * 60}\n")

# Query sul sistema
inference = VariableElimination(model)
print("Qual è la probabilità del carico se è notte e la fascia energetica sia la 1? - SITUAZIONE IMPOSSIBILE, silenzio della macchina")
q_notte = inference.query(variables=['target_carico'],
                          evidence={'hour_cat': 'Notte', 'fascia_oraria': 'F1'})
visualizza_risultati(q_notte)

print("Qual è la probabilità del carico se è pomeriggio, la fascia energetica sia la 1 e la media dei consumi sia basso?")
q_pomeriggio = inference.query(variables=['target_carico'],
                               evidence={'hour_cat': 'Pomeriggio', 'fascia_oraria': 'F1', 'media_mobile_cat': 'Trend_Basso'})
visualizza_risultati(q_pomeriggio)

print("Qual è la probabilità del carico se fa freddo e probabilmente piove, è mattina e normalmente si ha una media dei consumi alta?")
q_notte_f3 = inference.query(variables=['target_carico'],
                             evidence={'pioggia_cat': 'Prob_Alta', 'temp_cat':'Freddo', 'hour_cat':'Mattina', 'media_mobile_cat': 'Trend_Alto'})
visualizza_risultati(q_notte_f3)

print("Qual è la probabilità del carico se è mattina nella fascia 1 e la media dei consumi è alto?")
q_reale = inference.query(variables=['target_carico'],
                          evidence={'hour_cat':'Mattina', 'fascia_oraria': 'F1', 'media_mobile_cat': 'Trend_Alto'})
visualizza_risultati(q_reale)

Avvio Cross-Validation (5-Fold) per la Rete Bayesiana...


  0%|          | 0/471 [00:00<?, ?it/s]

Completato Fold 1/5: Accuratezza 90.70%


  0%|          | 0/475 [00:00<?, ?it/s]

Completato Fold 2/5: Accuratezza 90.90%


...

Completato Fold 3/5: Accuratezza 91.20%


  0%|          | 0/487 [00:00<?, ?it/s]

Completato Fold 4/5: Accuratezza 90.83%


  0%|          | 0/480 [00:00<?, ?it/s]

Completato Fold 5/5: Accuratezza 91.31%

------------------------------------------------------------
RISULTATI CROSS-VALIDATION RETE BAYESIANA: 
Accuratezza Media in CV (K-Fold=5): 90.98%
Deviazione Standard in CV: 0.23%
------------------------------------------------------------

Qual è la probabilità del carico se è notte e la fascia energetica sia la 1? - SITUAZIONE IMPOSSIBILE, silenzio della macchina
Stato Carico    | Probabilità
------------------------------
Basso           | 33.33%
Medio           | 33.33%
Alto            | 33.33%


Qual è la probabilità del carico se è pomeriggio, la fascia energetica sia la 1 e la media dei consumi sia basso?
Stato Carico    | Probabilità
------------------------------
Basso           | 90.44%
Medio           | 8.13%
Alto            | 1.43%


Qual è la probabilità del carico se fa freddo e probabilmente piove, è mattina e normalmente si ha una media dei consumi alta?
Stato Carico    | Probabilità
------------------------------
Basso        

**Parte 4** : Knowledge Base (Prolog)

In [21]:
prolog = Prolog()
with open("kb_smarthome.pl", "w") as f:
    f.write("""
    % Tipologia e Priorità
    tipo(frigo, critico).
    tipo(forno1, sacrificabile).
    tipo(forno2, sacrificabile).
    tipo(microonde, sacrificabile).
    tipo(lavastoviglie, sacrificabile).
    tipo(fornello, sacrificabile).

    priorita_distacco(lavastoviglie, 1). % 1 = Spegnere per primo
    priorita_distacco(fornello, 1).
    priorita_distacco(microonde, 2).
    priorita_distacco(forno2, 3).
    priorita_distacco(forno1, 4).

    % Trova e ordina per priorità i carichi attivi
    carichi_attivi_ordinati(ListaNomi) :-
        findall(P-E, (stato(E, acceso), tipo(E, sacrificabile), priorita_distacco(E, P)), ListaP),
        keysort(ListaP, ListaOrdinata),
        estrai_elettrodomestici(ListaOrdinata, ListaNomi).

    % Estrae solo i nomi dalla lista di coppie Priorita-Elettrodomestico attraverso la ricorsione
    estrai_elettrodomestici([], []).
    estrai_elettrodomestici([_-E|Resto], [E|NomiResto]) :-
        estrai_elettrodomestici(Resto, NomiResto).

    % MOTORE DI INFERENZA CON LOGICA DELLE FASCE ORARIE
    % Regola F1/F2 + Carico Alto: Focus su ECONOMIA
    consiglio(economia_alto, Messaggio) :-
        previsione_carico(alto),
        (fascia_oraria(f1) ; fascia_oraria(f2)),
        carichi_attivi_ordinati(Attivi),
        (Attivi = [] -> Messaggio = "ECONOMIA: Fascia costosa e previsione alta. Anche se non ci sono grandi carichi attivi, non accenderne di nuovi!"
        ;
        atomic_list_concat(Attivi, ' poi ', AttiviStr),
        string_concat("ECONOMIA: Fascia costosa e carico alto. Si consiglia la seguente sequenza di distacco: ", AttiviStr, Messaggio)).

    % Regola F3 + Carico Alto: Focus su SICUREZZA TECNICA (Rischio Blackout)
    consiglio(sicurezza_alto, Messaggio) :-
        previsione_carico(alto),
        fascia_oraria(f3),
        carichi_attivi_ordinati(Attivi),
        (Attivi = [] -> Messaggio = "SICUREZZA: Sei in fascia F3 (economica), ma la previsione di carico totale e' critica. Attenzione a non superare il limite del contatore."
        ;
        atomic_list_concat(Attivi, ' poi ', AttiviStr),
        string_concat("SICUREZZA TECNICA: Anche se sei in fascia F3, il carico e' troppo alto e rischi un blackout. Riduci subito l'uso di: ", AttiviStr, Messaggio)).

    % Regola Carico Medio + Rischio Storico Alto (Prudenza)
    consiglio(prudenza, Messaggio) :-
        previsione_carico(medio),
        pericolo_statistico_fascia(P), P > 0.30,
        Messaggio = "ATTENZIONE: Storicamente in questa fascia il rischio di picchi e' alto. Il carico ora e' MEDIO, evita di accendere nuovi elettrodomestici!".

    % Regole di Info e Ottimizzazione
    consiglio(info_frigo, "INFO: Il frigo e' in funzione e contribuisce al carico, ma essendo un dispositivo critico non verra' disattivato.") :-
        previsione_carico(alto),
        stato(frigo, acceso).

    consiglio(f3_ottimale, "OTTIMIZZAZIONE: Carico medio/basso in fascia F3. Stai gestendo perfettamente i tuoi consumi!") :-
        (previsione_carico(medio) ; previsione_carico(basso)),
        fascia_oraria(f3).

    consiglio(solare, "OTTIMIZZAZIONE: La produzione di energia solare e' alta al momento! Puoi sfruttarla per accendere nuovi carichi.") :-
        produzione_solare(alta), previsione_carico(basso).
    """)

prolog.consult("kb_smarthome.pl")

In [22]:
def aggiorna_prolog_senza_errori(riga):
    prolog.retractall("fascia(_)")
    prolog.retractall("stato(_, _)")
    prolog.retractall("previsione_carico(_)")

    f_raw = str(riga['fascia_oraria'].values[0]).lower().strip()
    prolog.assertz(f"fascia({f_raw})")

    traduzione_nomi = {
        "dishwasher": "lavastoviglie",
        "furnace1": "forno1",
        "furnace2": "forno2",
        "microwave": "microonde",
        "fridge": "frigo"
    }

    for col in riga.columns:
        if "[kW]" in col and col != 'use [kW]':
            valore = riga[col].values[0]
            nome_originale = col.replace(" [kW]", "").replace(" ", "").lower()
            nome_logico = traduzione_nomi.get(nome_originale, nome_originale)
            stato = "acceso" if valore > 0.1 else "spento"
            prolog.assertz(f"stato({nome_logico}, {stato})")

In [28]:
def ottieni_consigli_smart(modello_scelto, dati_per_modello, riga_sensori, nome_modello):
    mapping = {0: "basso", 1: "medio", 2: "alto"}

    try:
        if nome_modello == "Rete Bayesiana":
            dati_input = dati_per_modello.drop(columns=['target_carico'], errors='ignore')
            # Nelle versioni recenti di pgmpy predict_probability restituisce direttamente le probabilità
            prob_df = modello_scelto.predict_probability(dati_input)
            id_pred_raw = prob_df.idxmax(axis=1).values[0]
            id_pred = id_pred_raw.split('_')[-1] if '_' in str(id_pred_raw) else id_pred_raw
            label_pred = mapping[int(id_pred)]
            print(f"Per il modello {nome_modello}, la probabilità che il carico sia {label_pred.upper()} è del {prob_df.max(axis=1).values[0]*100:.2f}%")
        elif nome_modello == "Albero Decisionale":
            cols = modello_scelto.feature_names_in_
            pred_val = modello_scelto.predict(dati_per_modello[cols])[0]
            label_pred = mapping[int(pred_val)]
            print(f"Il modello {nome_modello} predice che il carico sia {label_pred.upper()}")
        else: # Rete Neurale MLP
            cols = X_train.columns
            pred_val = modello_scelto.predict(dati_per_modello[cols].values)[0]
            label_pred = mapping[int(pred_val)]
            print(f"Il modello {nome_modello} predice che il carico sia {label_pred.upper()}")
    except Exception as e:
        print(f"Errore {nome_modello}: {e}")
        return

    # Aggiornamento i fatti dinamici (Stato elettrodomestici)
    aggiorna_prolog_senza_errori(riga_sensori)

    # Controllo riguardo la storicità delle fasce
    fascia_corrente = str(riga_sensori['fascia_oraria'].values[0]).lower().strip()
    try:
        prob_alto_storica = distribuzione.loc[riga_sensori['fascia_oraria'].values[0], 'Carico Alto']
    except:
        prob_alto_storica = 0
    prolog.assertz(f"pericolo_statistico_fascia({prob_alto_storica})")

    # Controllo produzione solare
    valore_solare = riga_sensori['Solar [kW]'].values[0]
    solare_status = "alta" if valore_solare > 0.4 else "bassa"
    prolog.assertz(f"produzione_solare({solare_status})")

    # Asseriamo carico e fascia
    prolog.assertz(f"previsione_carico({label_pred})")
    prolog.assertz(f"fascia_oraria({fascia_corrente})")

    # Interroghiamo la Knowledge Base
    res = list(prolog.query("consiglio(ID, Messaggio)"))
    if res:
        messaggi_unici = set()
        for r in res:
            msg = r['Messaggio'].decode('utf-8') if isinstance(r['Messaggio'], bytes) else r['Messaggio']
            messaggi_unici.add(f"  -> {msg}")
        for m in messaggi_unici:
            print(m)
    else:
        print("  -> Sistema in equilibrio. Nessun distacco o avviso necessario.")

    return label_pred

In [25]:
def test_comparativo_modelli(indice_scenario):
    riga_originale = df.iloc[[indice_scenario]]
    riga_bayes = df_bayes.iloc[[indice_scenario]]

    print(f"\nANALISI SCENARIO: {indice_scenario}")
    print(f"Carico Misurato: {df['use [kW]'].iloc[indice_scenario]:.4f} kW")

    modelli = [
        ("Albero Decisionale", best_dt, riga_originale),
        ("Rete Neurale (MLP)", best_mlp, riga_originale),
        ("Rete Bayesiana", model, riga_bayes)
    ]

    risultati_individuali = []
    for nome, mod, dati in modelli:
        pred_label = ottieni_consigli_smart(mod, dati, riga_originale, nome)
        risultati_individuali.append(pred_label)
        print("-" * 40)

    decisione_integrata, motivo = predizione_integrata_voto(
        risultati_individuali[0],
        risultati_individuali[1],
        risultati_individuali[2]
    )

    print(f"\nEsito del voto: {motivo}")

    ragionamento_kb_finale(decisione_integrata, riga_originale)

In [23]:
def predizione_integrata_voto(pred_albero, pred_mlp, pred_bayes):
    mappa = {0: "basso", 1: "medio", 2: "alto"}
    p_a = mappa.get(pred_albero, pred_albero)
    p_m = mappa.get(pred_mlp, pred_mlp)
    p_b = mappa.get(pred_bayes, pred_bayes)
    voti = [p_a, p_m, p_b]

    decisione = max(set(voti), key=voti.count)
    if voti.count(decisione) >= 2:
        motivo = f"Voto di Maggioranza democratica ({voti.count(decisione)} su 3 modelli concordano su {decisione.upper()})"
    else:
        # Parità (1 voto basso, 1 medio, 1 alto)
        decisione = "alto"
        motivo = "Stallo totale (1 voto per classe). Attivato Fallback di Sicurezza su ALTO."

    return decisione, motivo

In [26]:
def ragionamento_kb_finale(decisione_voto, riga_sensori):
    print("\n" + "="*80)
    print(f">>> ANALISI INTEGRATA KB FINALE <<<")
    print(f"Decisione tipologia del carico (Voto Maggioranza): {decisione_voto.upper()}")

    prolog.retractall("previsione_carico(_)")
    prolog.retractall("fascia_oraria(_)")
    prolog.retractall("produzione_solare(_)")
    prolog.retractall("pericolo_statistico_fascia(_)")

    prolog.assertz(f"previsione_carico({decisione_voto})")

    fascia_corrente = str(riga_sensori['fascia_oraria'].values[0]).lower().strip()
    prolog.assertz(f"fascia_oraria({fascia_corrente})")

    valore_solare = riga_sensori['Solar [kW]'].values[0]
    solare_status = "alta" if valore_solare > 0.4 else "bassa"
    prolog.assertz(f"produzione_solare({solare_status})")

    try:
        prob_alto_storica = distribuzione.loc[str(riga_sensori['fascia_oraria'].values[0]), 'Carico Alto']
    except:
        prob_alto_storica = 0
    prolog.assertz(f"pericolo_statistico_fascia({prob_alto_storica})")

    print(f"\nConsigli del sistema esperto (Fascia {fascia_corrente.upper()}):")
    res_finale = list(prolog.query("consiglio(ID, Messaggio)"))

    if not res_finale:
        print("Nessuna anomalia rilevata. Il sistema opera nei parametri ottimali.")
    else:
        visti = set()
        for r in res_finale:
            msg = r['Messaggio'].decode('utf-8') if isinstance(r['Messaggio'], bytes) else r['Messaggio']

            if msg not in visti:
                print(f"[*] {msg}")
                visti.add(msg)
    print("="*80 + "\n")

In [29]:
while True:
  try:
    n = int(input("Seleziona scenario (da 1 a 503.909) - 0 per uscire: "))
  except ValueError:
    print("Valore non valido. Riprova.")
    continue
  if n > 0 and n < 503910 :
    test_comparativo_modelli(n)
  elif n == 0:
    print("Grazie per aver usato il sistema intelligente EnergySafe!")
    break
  else:
    print("Indice scenario non esistente. Riprovarne un altro fra 1 e 509.909")

Seleziona scenario (da 1 a 503.909) - 0 per uscire: 43546

ANALISI SCENARIO: 43546
Carico Misurato: 0.4607 kW
Il modello Albero Decisionale predice che il carico sia MEDIO
  -> Sistema in equilibrio. Nessun distacco o avviso necessario.
----------------------------------------
Il modello Rete Neurale (MLP) predice che il carico sia MEDIO
  -> Sistema in equilibrio. Nessun distacco o avviso necessario.
----------------------------------------
Per il modello Rete Bayesiana, la probabilità che il carico sia MEDIO è del 90.36%
  -> Sistema in equilibrio. Nessun distacco o avviso necessario.
----------------------------------------

Esito del voto: Voto di Maggioranza democratica (3 su 3 modelli concordano su MEDIO)

>>> ANALISI INTEGRATA KB FINALE (SISTEMA ESPERTO) <<<
Decisione tipologia del carico (Voto Maggioranza): MEDIO

Consigli del sistema esperto (Fascia F1):
Nessuna anomalia rilevata. Il sistema opera nei parametri ottimali.

Seleziona scenario (da 1 a 503.909) - 0 per uscire: 312